# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2

import logging, sys
logging.basicConfig(level=logging.DEBUG)

import molpot as mpot
import torch
from molpot import alias

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [15]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    preprocess=[
        mpot.pipline.NeighborList(cutoff=5.0),
        # mpot.pipline.AtomicDressing(mpot.dataset.rMD17.atomrefs)
    ],
)

INFO:molpot:Parsing molecule aspirin
INFO:molpot:Loaded 1000 frames


In [3]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                 dataset: rmd17-aspirin                 
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [4]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.95, .05])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=False,
)
eval_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=2,
    shuffle=False,
)

## Setting up the model

In [5]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 4.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(4.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1
)
e_readout = mpot.potential.nnp.readout.Atomwise(
    [64, 1],
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy")
)
f_readout = mpot.potential.nnp.readout.Derivative(
    fx_key=("predicts", "energy"),
    dx_key=("atoms", "R"),
    to_key=("predicts", "forces")
)
model = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)

In [6]:
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

loss_fn = mpot.MultiTargetLoss(torch.nn.MSELoss(), [("energy", "energy", 1.0)])

trainer = mpot.PotentialTrainer(
    model,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-4),
    loss_fn=loss_fn,
    device="cuda",
)
trainer.add_evaluator(eval_dl, max_epochs=2, epoch_length=10)

def get_energy(data):
    return (data[0]["energy"], data[1]["energy"])

def get_forces(data):
    return (data[0]["forces"], data[1]["forces"])

trainer.add_metric(
    "e_mae",
    MeanAbsoluteError(output_transform=get_energy),
    target="all",
    usage=BatchWise(),
)
trainer.add_metric(
    "f_mae",
    MeanAbsoluteError(output_transform=get_forces),
    target="all",
    usage=BatchWise(),
)

trainer.attach_progressbar()
trainer.attach_tensorboard(log_dir=Path("rmd17_log"))
# basic_profiler = trainer.attach_basic_time_profiler()
# handler_profiler = trainer.attach_handlers_time_profiler()

# trainer.run_tensorboard_profiler(train_dl, log_dir=Path("rmd17_profile"))

# state = trainer.run(train_dl, max_epochs=2, epoch_length=10)
# basic_profiler.print_results(basic_profiler.get_results())
# handler_profiler.print_results(handler_profiler.get_results())
# state

DEBUG:ignite.engine.engine.Engine:Added handler for event Events.EPOCH_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_STARTED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_STARTED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_STARTED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_STARTED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_CO

In [21]:
trainer.reset()
state = trainer.run(train_dl, 10)
state.metrics

INFO:ignite.engine.engine.Engine:Engine run starting with max_epochs=10.
DEBUG:ignite.engine.engine.Engine:0 | 0, Firing handlers for event Events.STARTED
DEBUG:ignite.engine.engine.Engine:1 | 0, Firing handlers for event Events.EPOCH_STARTED
DEBUG:ignite.engine.engine.Engine:1 | 0, Firing handlers for event Events.GET_BATCH_STARTED


/opt/conda/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: neighbors::getNeighborPairs: an autograd kernel was not registered to the Autograd key(s) but we are trying to backprop through it. This may lead to silently incorrect behavior. This behavior is deprecated and will be removed in a future version of PyTorch. If your operator is differentiable, please ensure you have registered an autograd kernel to the correct Autograd key (e.g. DispatchKey::Autograd, DispatchKey::CompositeImplicitAutograd). If your operator is not differentiable, or to squash this warning and use the previous behavior, please register torch::CppFunction::makeFallthrough() to DispatchKey::Autograd. (Triggered internally at ../torch/csrc/autograd/autograd_not_implemented_fallback.cpp:62.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
DEBUG:ignite.engine.engine.Engine:1 | 0, Firing handlers for event Events.GET_BATCH_COMPLETED
DEBUG:i

RuntimeError: One of the differentiated Tensors appears to not have been used in the graph. Set allow_unused=True if this is the desired behavior.

In [ ]:
mpot.inspector.ModelInspector(model)

In [ ]:
from molpot_op import get_neighbor_pairs
xyz = torch.rand(10, 3, requires_grad=True)
out = get_neighbor_pairs(xyz, 0.5, -1, None)

In [ ]:
out[1].requires_grad

True

In [10]:
rmd17_ds.frames[0]['pairs']['diff'].requires_grad

True

In [12]:
from torch.autograd import grad
grad(
    torch.sum(rmd17_ds.frames[0]['pairs']['diff']),
    rmd17_ds.frames[0]['atoms']['R'],
)

/opt/conda/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: neighbors::getNeighborPairs: an autograd kernel was not registered to the Autograd key(s) but we are trying to backprop through it. This may lead to silently incorrect behavior. This behavior is deprecated and will be removed in a future version of PyTorch. If your operator is differentiable, please ensure you have registered an autograd kernel to the correct Autograd key (e.g. DispatchKey::Autograd, DispatchKey::CompositeImplicitAutograd). If your operator is not differentiable, or to squash this warning and use the previous behavior, please register torch::CppFunction::makeFallthrough() to DispatchKey::Autograd. (Triggered internally at ../torch/csrc/autograd/autograd_not_implemented_fallback.cpp:62.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


(tensor([[-16., -16., -16.],
         [-16., -16., -16.],
         [-12., -12., -12.],
         [ -8.,  -8.,  -8.],
         [-10., -10., -10.],
         [ -9.,  -9.,  -9.],
         [ -8.,  -8.,  -8.],
         [ -5.,  -5.,  -5.],
         [ -2.,  -2.,  -2.],
         [  2.,   2.,   2.],
         [  2.,   2.,   2.],
         [  5.,   5.,   5.],
         [  5.,   5.,   5.],
         [  9.,   9.,   9.],
         [  9.,   9.,   9.],
         [  9.,   9.,   9.],
         [  9.,   9.,   9.],
         [ 10.,  10.,  10.],
         [  8.,   8.,   8.],
         [  8.,   8.,   8.],
         [ 10.,  10.,  10.]]),)